## RAG Pipeline - Data Ingestion to vector DB Pipeline

In [20]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
#  Modern / Correct
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [21]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: AdventorsAI_Offer_Letter_Kartik_Kukde.pdf
  ✓ Loaded 6 pages

Processing: attention.pdf
  ✓ Loaded 15 pages

Processing: RAG_nlp.pdf
  ✓ Loaded 19 pages

Total documents loaded: 40


In [22]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'AdventorsAI_Offer_Letter_Kartik_Kukde.docx', 'source': '..\\data\\pdf\\AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'file_type': 'pdf'}, page_content="OFFER  LETTER  \nTo,\n \n \n \n \n \n \n \n \n \nDate:\n \n29\n \nJuly\n \n2026\n \nMr.  Kartik  Kukde  \nSubject:  Offer  of  Engagement  as  Technical  Specialist  Intern  \nDear\n \nMr.\n \nKartik\n \nKukde,\n \n \nWe\n \nare\n \npleased\n \nto\n \noffer\n \nyou\n \nthe\n \nposition\n \nof\n \nTechnical\n \nSpecialist\n \nIntern\n \nwith\n \nAdventors.AI,\n \neffective\n \n03\n \nAugust\n \n2026.\n \n \nAdventors.AI\n \nis\n \na\n \nFuture-Ready\n \nSchool\n \nTransformation\n \nCompany\n \ncommitted\n \nto\n \nempowering\n \neducational\n \ninstitutions\n \nthrough\n \nArtificial\n \nIntelligence,\n \nCoding,\n

In [23]:
## Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    "split documents into smaller chunks for the better RAG performance"
    text_splitter= RecursiveCharacterTextSplitter(
         chunk_size=chunk_size,
         chunk_overlap=chunk_overlap,
         length_function=len,
         separators=["\n\n","\n"," ",""]
     )
    split_docs = text_splitter.split_documents(documents)
    print(f"split (len(documents)) documents into (len(split_docs)) chunks")
     
     #show example of a chunk
    if split_docs:
         print(f"\nExample chunk:")
         print(f"content: {split_docs[0].page_content[:200]}...")
         print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
     


In [24]:
chunks = split_documents(all_pdf_documents)
print(chunks)

split (len(documents)) documents into (len(split_docs)) chunks

Example chunk:
content: OFFER  LETTER  
To,
 
 
 
 
 
 
 
 
 
Date:
 
29
 
July
 
2026
 
Mr.  Kartik  Kukde  
Subject:  Offer  of  Engagement  as  Technical  Specialist  Intern  
Dear
 
Mr.
 
Kartik
 
Kukde,
 
 
We
 
are
 
p...
Metadata: {'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'AdventorsAI_Offer_Letter_Kartik_Kukde.docx', 'source': '..\\data\\pdf\\AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'AdventorsAI_Offer_Letter_Kartik_Kukde.docx', 'source': '..\\data\\pdf\\AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'AdventorsAI_Offer_Letter_Kartik_Kukde.pdf', 'file_type'

embedding & vectorStoreDB

In [25]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
import os
import warnings

# Disable Hugging Face symlink warning on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Suppress general UserWarnings from file_download
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

# Now import sentence-transformers
from sentence_transformers import SentenceTransformer
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1139.28it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\kartik\AppData\Local\Temp\ipykernel_15696\4013185584.py:31: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Vectorstore

In [27]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 20


In [28]:
##converting the text to embeddings
texts = [doc.page_content for doc in chunks]

##generating the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

##storing the embeddings into the vecotor database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 164 texts...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches: 100%|██████████| 6/6 [00:20<00:00,  3.50s/it]


Generated embeddings with shape: (164, 384)
Adding 164 documents to vector store...
Successfully added 164 documents to vector store
Total documents in collection: 184


Retriever Pipeline from vectorstore

In [29]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [31]:
rag_retriever.retrieve("Jeopardy question generation")

Retrieving documents for query: 'Jeopardy question generation'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.57it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_fbf76985_94',
  'content': 'Jeopardy is an unusual format that consists of trying to guess an entity from a fact about that entity.\nFor example, “The World Cup” is the answer to the question “In 1986 Mexico scored as the ﬁrst\ncountry to host this international sports competition twice.” As Jeopardy questions are precise,\nfactual statements, generating Jeopardy questions conditioned on their answer entities constitutes a\nchallenging knowledge-intensive generation task.\nWe use the splits from SearchQA [ 10], with 100K train, 14K dev, and 27K test examples. As\nthis is a new task, we train a BART model for comparison. Following [67], we evaluate using the\nSQuAD-tuned Q-BLEU-1 metric [ 42]. Q-BLEU is a variant of BLEU with a higher weight for\nmatching entities and has higher correlation with human judgment for question generation than\nstandard metrics. We also perform two human evaluations, one to assess generation factuality, and',
  'metadata': {'file_type': 'pdf',
 

Integration of vectordb context pipeline eith LLM output

In [32]:

import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

<GROQ_API_KEY_HIDDEN>


In [41]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [42]:
answer=rag_simple("What does RAG sequence model use to generate the complete sequence?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What does RAG sequence model use to generate the complete sequence?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The RAG‑Sequence model treats the retrieved document(s) as a single latent variable and uses the generator pθ to produce the full output sequence for each retrieved passage, then marginalises over the top‑K retrieved documents to obtain the final sequence.



Enhanced RAG Pipeline Features

In [55]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How many identical layers make up the encoder stack, and what is the output dimension?", rag_retriever, llm, top_k=3, min_score=-2.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'How many identical layers make up the encoder stack, and what is the output dimension?'
Top K: 3, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 14.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The encoder stack consists of **6 identical layers**, and each layer (and thus the encoder’s output) has a dimension of **d model = 512**.
Sources: [{'source': 'attention.pdf', 'page': 2, 'score': 0.07562965154647827, 'preview': 'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder...'}, {'source': 'attention.pdf', 'page': 2, 'score': -0.028461217880249023, 'preview': 'itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder in...'}, {'source': 'attention.pdf', 'page': 5, 'score': -0.061618

In [52]:
test_queries = [
    # 1. Architecture hyperparameter (Section 3.1)
    "How many identical layers make up the encoder stack, and what is the output dimension?",
    # 2. Optimization settings (Section 5.3)
    "What are the values of beta1, beta2, and epsilon for the Adam optimizer used in the paper?",
    # 3. Regularization setting (Section 5.4)
    "What value of label smoothing was used during training, and how did it affect perplexity?",
]

for i, q in enumerate(test_queries, 1):
    print(f"\n{'='*70}\n[Test {i}] Query: {q}\n{'='*70}")
    res = rag_advanced(
        q,
        rag_retriever,
        llm,
        top_k=3,
        min_score=-2.0,  # permissive threshold to inspect raw behavior
        return_context=True,
    )
    print("Answer:\n", res["answer"])
    print("\nConfidence Score:", res["confidence"])
    print("Retrieved Sources:")
    for s in res["sources"]:
        print(f"  - Page: {s['page']} | Score: {s['score']}")


[Test 1] Query: How many identical layers make up the encoder stack, and what is the output dimension?
Retrieving documents for query: 'How many identical layers make up the encoder stack, and what is the output dimension?'
Top K: 3, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer:
 The encoder stack consists of **6 identical layers**, and each layer (and thus the encoder’s output) has a dimension of **d model = 512**.

Confidence Score: 0.07562965154647827
Retrieved Sources:
  - Page: 2 | Score: 0.07562965154647827
  - Page: 2 | Score: -0.028461217880249023
  - Page: 5 | Score: -0.061618685722351074

[Test 2] Query: What are the values of beta1, beta2, and epsilon for the Adam optimizer used in the paper?
Retrieving documents for query: 'What are the values of beta1, beta2, and epsilon for the Adam optimizer used in the paper?'
Top K: 3, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer:
 The Adam optimizer was configured with  

- **β₁ = 0.9**  
- **β₂ = 0.98**  
- **ε = 10⁻⁹**.

Confidence Score: -0.07855308055877686
Retrieved Sources:
  - Page: 6 | Score: -0.07855308055877686
  - Page: 10 | Score: -0.3513718843460083
  - Page: 0 | Score: -0.35298073291778564

[Test 3] Query: What value of label smoothing was used during training, and how did it affect perplexity?
Retrieving documents for query: 'What value of label smoothing was used during training, and how did it affect perplexity?'
Top K: 3, Score threshold: -2.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer:
 The models were trained with label smoothing εₗₛ = 0.1. This smoothing raises (i.e., hurts) the perplexity because the model is encouraged to be less certain, even though it improves accuracy and BLEU.

Confidence Score: -0.3885080814361572
Retrieved Sources:
  - Page: 7 | Score: -0.3885080814361572
  - Page: 8 | Score: -0.44657421112060547
  - Page: 7 | Score: -0.4734383821487427
